In [1]:
import os
import json
import pandas as pd
import traceback

# ignore warnings
import warnings
warnings.filterwarnings("ignore")


In [2]:
# from langchain.chat_models import ChatOpenAI
from langchain_community.chat_models import ChatOpenAI

In [7]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

True

In [8]:
KEY=os.getenv("OPENAI_API_KEY")
llm=ChatOpenAI(openai_api_key=KEY, model_name="gpt-4o-mini", temperature=0.2)

In [11]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_community.callbacks.manager import get_openai_callback
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
import json

In [12]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [13]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [14]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [20]:
quiz_chain=LLMChain(llm=llm,prompt=quiz_generation_prompt,output_key="quiz",verbose=True)


In [21]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [22]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"], template=TEMPLATE)

In [23]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [24]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [25]:
file_path=r"/workspaces/mcqgen/data.txt"
file_path

'/workspaces/mcqgen/data.txt'

In [26]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [27]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [28]:
NUMBER=5 
SUBJECT="Galaxy Z Flip6"
TONE="Easy"

In [30]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:What's new and different about the <Galaxy Z Flip6> ? 	"Would you believe it if I told you there's a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!

The Z Flip6's 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.
Additionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

Text:What's new and different about the <Galaxy Z Flip6> ? 	"Would you believe it if I told you there's a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!

The Z Flip6's 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.
Additionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy more vivid LED effects

In [31]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost:.3f}")

Total Tokens:5519
Prompt Tokens:3174
Completion Tokens:2345
Total Cost:0.002


In [32]:
response

{'text': 'What\'s new and different about the <Galaxy Z Flip6> ? \t"Would you believe it if I told you there\'s a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!\n\nThe Z Flip6\'s 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.\nAdditionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy more vivid LED effects on the screen with interactive motion content by attaching LED Effe

In [37]:
quiz = response.get('quiz')
print(quiz)
quiz = json.loads(quiz)

```JSON
{
  "1": {
    "mcq": "What is a new feature of the Galaxy Z Flip6 that enhances selfies?",
    "options": {
      "a": "AI zoom",
      "b": "Night mode",
      "c": "3D effects",
      "d": "All of the above"
    },
    "correct": "d"
  },
  "2": {
    "mcq": "How many colors does the Galaxy Z Flip6 come in?",
    "options": {
      "a": "3",
      "b": "4",
      "c": "7",
      "d": "5"
    },
    "correct": "b"
  },
  "3": {
    "mcq": "What is the purpose of the Interpreter feature in the Galaxy Z Flip6?",
    "options": {
      "a": "To edit photos",
      "b": "To translate conversations",
      "c": "To enhance video quality",
      "d": "To customize wallpapers"
    },
    "correct": "b"
  },
  "4": {
    "mcq": "Which of the following is a color exclusive to Samsung.com?",
    "options": {
      "a": "Mint",
      "b": "Crafted Black",
      "c": "Silver Shadow",
      "d": "Blue"
    },
    "correct": "b"
  },
  "5": {
    "mcq": "What material improvements have bee

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [36]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

AttributeError: 'str' object has no attribute 'items'

In [ ]:
quiz_table_data

[{'MCQ': "What is a key feature of the Galaxy Z Flip6's camera?",
  'Choices': 'a: 50 MP flagship camera | b: 30 MP camera | c: No camera | d: 20 MP camera',
  'Correct': 'a'},
 {'MCQ': 'Which new feature helps in translating conversations on the Galaxy Z Flip6?',
  'Choices': 'a: Chat Assist | b: Interpreter | c: FlexWindow | d: Zoom Rocker',
  'Correct': 'b'},
 {'MCQ': 'What is the primary function of the Camcorder grip on the Z Flip6?',
  'Choices': 'a: To take selfies | b: To record videos easily | c: To call someone | d: To play music',
  'Correct': 'b'},
 {'MCQ': 'Which color is NOT mentioned as one of the standard colors for the Galaxy Z Flip6?',
  'Choices': 'a: Blue | b: Yellow | c: Mint | d: Silver Shadow',
  'Correct': 'b'},
 {'MCQ': 'What does the Galaxy AI feature allow users to do?',
  'Choices': 'a: Make calls | b: Edit images and generate portraits | c: Play games | d: Browse the internet',
  'Correct': 'b'},
 {'MCQ': 'Which exclusive colors are available only on Samsun

In [ ]:
quiz = pd.DataFrame(quiz_table_data)

In [ ]:
quiz.to_csv("GalaxyZFlip6.csv",index=False)